# TOMAS-JAX in AMBRS

[TOMAS-JAX](https://github.com/reflective-org/tomas-jax) is a JAX implementation of the
TwO-Moment Aerosol Sectional model: it tracks number *and* per-species mass in each of 40
mass-doubling size bins, from about 1.7 nm to 17.5 µm.

Unlike PartMC and MAM4 it is a Python library rather than a compiled executable, so AMBRS
steps it **in process**. That means this notebook runs with no box-model binaries built —
`pip install -r requirements.txt` is enough.

What follows:

1. define a scenario,
2. run it and watch the size distribution evolve,
3. see what each microphysical process contributes,
4. compare the sectional representation against the modal one MAM4 uses,
5. notes on comparing against PartMC and MAM4 dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import ambrs
import ambrs.aerosol as aerosol
import ambrs.gas as gas

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,          # recessive grid
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# a fixed, colourblind-safe order (Okabe-Ito); assigned in order, never cycled
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#E69F00"]

# Evaluate every distribution on TOMAS's own 40 bins. Sampling more finely than the
# model resolves would leave empty bins between populated ones, which draws as a comb
# of spikes rather than a distribution.
from tomas_jax.core.config import xk_boundaries

DIAMETERS = ambrs.tomas_jax.bin_diameters(xk_boundaries())   # [m], 40 bins
PLOT_RANGE_NM = (2.0, 3000.0)  # the grid reaches 15 um, but nothing lives out there


def size_distribution(output, diameters=DIAMETERS):
    """dN/dlnD [# m^-3] for an ambrs Output, on the given diameter grid [m]."""
    return np.asarray(output.compute_variable("dNdlnD", {
        "diam_grid": diameters,
        "normalize": False,
        "wetsize": False,
        "method": "hist",
    }))


def style_axes(ax, title):
    ax.set_xscale("log")
    ax.set_xlim(*PLOT_RANGE_NM)
    ax.set_xlabel("dry diameter [nm]")
    ax.set_ylabel("dN/dlnD [# m$^{-3}$]")
    ax.set_title(title)


def summarise(label, output):
    population = output.particle_population
    return {
        "case": label,
        "N [# m^-3]": population.get_Ntot(),
        "dry mass [kg m^-3]": population.get_tot_dry_mass(),
    }


print("tomas-jax available:", ambrs.tomas_jax._TOMAS_AVAILABLE)

## 1. Define a scenario

A `Scenario` is model-agnostic: the same object can be handed to any AMBRS box model. Here
it's a sulfate aerosol with an Aitken and an accumulation mode, plus gas-phase H2SO4 for
condensation to act on.

In [ ]:
so4 = aerosol.AerosolSpecies(name="SO4", molar_mass=97.071, density=1770,
                             hygroscopicity=0.507)
h2so4 = gas.GasSpecies(name="H2SO4", molar_mass=98.079)


def mode(name, number, geom_mean_diam, gsd):
    return aerosol.AerosolModeState(
        name=name, species=(so4,),
        number=number,                        # [# m^-3]
        geom_mean_diam=geom_mean_diam,        # [m]
        log10_geom_std_dev=np.log10(gsd),
        mass_fractions=(1.0,))


scenario = ambrs.Scenario(
    aerosols=(so4,),
    gases=(h2so4,),
    size=aerosol.AerosolModalSizeState(modes=(
        mode("aitken", 5e11, 3.0e-8, 1.6),
        mode("accumulation", 1e11, 1.2e-7, 1.6),
    )),
    gas_concs=(5e7,),          # H2SO4 [molec cm^-3]
    flux=0.0,
    relative_humidity=0.5,
    temperature=288.0,         # [K]
    pressure=101325.0,         # [Pa]
    height=500.0,              # [m]
)

for m in scenario.size.modes:
    print(f"{m.name:>14}: N = {m.number:.2e} /m3, GMD = {m.geom_mean_diam*1e9:6.1f} nm, "
          f"GSD = {10**m.log10_geom_std_dev:.2f}")

## 2. Run it, and watch the distribution evolve

Coagulation and condensation together, over 24 simulated hours. `run_ensemble` compiles the
solver once and reuses it across every input, so the JIT cost is paid a single time.

In [ ]:
model = ambrs.tomas_jax.AerosolModel(
    ambrs.AerosolProcesses(coagulation=True, condensation=True),
    h2so4_production=5e4,      # H2SO4 source [molec cm^-3 s^-1]
)

DT = 60.0                                     # [s]
HOURS = [0, 1, 6, 24]
inputs = [model.create_input(scenario, dt=DT, nstep=int(h * 3600 / DT) or 1)
          for h in HOURS]
# nstep must be >= 1, so "hour 0" is a single step: effectively the initial state
outputs = model.run_ensemble(inputs)

fig, ax = plt.subplots()
for colour, hours, output in zip(PALETTE, HOURS, outputs):
    ax.plot(DIAMETERS * 1e9, size_distribution(output), lw=2,
            color=colour, label=f"{hours} h")
style_axes(ax, "Size distribution over 24 h (coagulation + condensation)")
ax.legend(title="elapsed", frameon=False)
plt.tight_layout()
plt.show()

import pandas as pd
pd.DataFrame([summarise(f"{h} h", o) for h, o in zip(HOURS, outputs)]).set_index("case")

Number falls as coagulation merges particles, while dry mass grows as H2SO4 condenses onto
them — the two-moment scheme tracks both independently.

## 3. What each process contributes

Six hours with different processes enabled — a like-for-like comparison, since each model
gets its own `AerosolProcesses` and nothing else changes.

This uses a *clean* background rather than the polluted scenario above, because nucleation
is strongly suppressed by a condensation sink: with the aerosol loading of section 1, fresh
clusters are scavenged as fast as they form and enabling nucleation changes the number
concentration by well under a percent. Thinning the background makes the effect visible —
which is itself the lesson.

In [ ]:
CASES = [
    ("coagulation only", ambrs.AerosolProcesses(coagulation=True)),
    ("+ condensation", ambrs.AerosolProcesses(coagulation=True, condensation=True)),
    ("+ nucleation", ambrs.AerosolProcesses(coagulation=True, condensation=True,
                                            nucleation=True)),
]

# a clean background with plenty of precursor: a nucleation-friendly airmass
clean_scenario = ambrs.Scenario(
    aerosols=(so4,), gases=(h2so4,),
    size=aerosol.AerosolModalSizeState(modes=(mode("background", 1e9, 1.2e-7, 1.6),)),
    gas_concs=(1e9,),          # H2SO4 [molec cm^-3]
    flux=0.0, relative_humidity=0.5, temperature=288.0,
    pressure=101325.0, height=500.0,
)

nstep = int(6 * 3600 / DT)
process_outputs = []
for label, processes in CASES:
    case_model = ambrs.tomas_jax.AerosolModel(
        processes,
        h2so4_production=1e7,  # H2SO4 source [molec cm^-3 s^-1]
        # precursors for the nucleation scheme; a Scenario doesn't carry these
        org_conc=1e9,          # condensable organics [molec cm^-3]
        nh3_conc=1e10,         # ammonia [molec cm^-3]
        fion=2.0,              # ion pairs [cm^-3 s^-1]
    )
    process_outputs.append(
        case_model.run(case_model.create_input(clean_scenario, dt=DT, nstep=nstep), label))

fig, ax = plt.subplots()
for colour, (label, _), output in zip(PALETTE, CASES, process_outputs):
    ax.plot(DIAMETERS * 1e9, size_distribution(output), lw=2, color=colour, label=label)
style_axes(ax, "Contribution of each process after 6 h (clean background)")
ax.set_yscale("log")   # nucleation changes number by orders of magnitude
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame([summarise(label, o)
              for (label, _), o in zip(CASES, process_outputs)]).set_index("case")

Coagulation alone only removes particles. Condensation grows them without changing their
number. Nucleation is the one process that *creates* particles: fresh ~1.7 nm clusters
appear at the left-hand edge of the grid, and here they raise the total number by orders of
magnitude. How many survive is then a competition between nucleation and the coagulation
that scavenges them.

## 4. Sectional against modal

TOMAS resolves the distribution in 40 mass-doubling bins. MAM4 instead represents it as a
sum of log-normal modes, which AMBRS turns into a particle population via part2pop's
`binned_lognormals` builder — the same call `mam4.retrieve_model_state` makes for its
initial state.

Both are built here from the *same* `Scenario`, so the difference is purely one of
representation, before any physics runs.

In [ ]:
from part2pop import build_population
from ambrs.analysis import Output

# the modal representation, exactly as mam4.retrieve_model_state builds it
modal_population = build_population({
    "type": "binned_lognormals",
    "D_min": 1e-9, "D_max": 1e-4, "N_bins": 200,
    "N": [m.number for m in scenario.size.modes],
    "GMD": [m.geom_mean_diam for m in scenario.size.modes],
    "GSD": [10 ** m.log10_geom_std_dev for m in scenario.size.modes],
    "aero_spec_names": [[s.name for s in m.species] for m in scenario.size.modes],
    "aero_spec_fracs": [m.mass_fractions for m in scenario.size.modes],
})
modal = Output(model_name="modal (as MAM4 represents it)", scenario_name="initial",
               scenario=scenario, timestep=1, particle_population=modal_population,
               gas_mixture=None, thermodynamics={})

# the sectional representation, straight off the TOMAS grid
sectional = model.run(model.create_input(scenario, dt=DT, nstep=1), "initial")

fig, ax = plt.subplots()
ax.plot(DIAMETERS * 1e9, size_distribution(modal), lw=2, color=PALETTE[0],
        label="modal (MAM4-style)")
ax.plot(DIAMETERS * 1e9, size_distribution(sectional), lw=2, color=PALETTE[1],
        label="sectional (TOMAS)")
style_axes(ax, "Same scenario, two representations")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame([summarise("modal", modal),
              summarise("sectional", sectional)]).set_index("case")

The modal curve is smooth by construction; the sectional one is resolved onto 40 bins, so it
is coarser at the edges but is free to take on shapes a sum of log-normals cannot — which is
the whole reason to compare the two.

## 5. Comparing against PartMC and MAM4

Those models are compiled executables, so they need binaries built with
[ambuilder](https://github.com/AMBRS-project/ambuilder) and are run through `PoolRunner`
rather than in process:

```python
runner = ambrs.PoolRunner(mam4_model, executable="mam4", root="mam4_runs")
runner.run(mam4_model.create_inputs(ensemble, dt=DT, nstep=nstep))

output = ambrs.mam4.retrieve_model_state(
    "1", scenario, timestep=nstep, ensemble_output_dir="mam4_runs")
```

Every model returns the same `ambrs.analysis.Output`, so once you have one from each, the
plotting above works unchanged — `size_distribution(output)` doesn't care which model
produced it. `ambrs.analysis.kl_divergence(a, b)` and `nmae([a], [b], "dNdlnD")` quantify
the difference between two of them.